# Day 70 — Phase 2 Comprehensive Examination & Mastery Review

This notebook covers First-Principles Architecture & Theory, Imbalanced Classification Metrics Audit, End-to-End ML System Design, and Phase 2 Capstone Coding Challenge (Random Forest & 5-Fold Cross-Validation)

## Validation goals
- Complete Part 1: Demonstrate mastery of first-principles architecture, mathematical bias-variance decomposition, sigmoid link functions in logistic regression, and Gini impurity node splitting
- Complete Part 2: Audit model evaluation metrics for high-imbalance classification tasks (fraud detection) and analyze Precision-Recall operational trade-offs
- Complete Part 3: Articulate enterprise ML system design principles covering data leakage prevention, residual-based uncertainty quantification, and model explainability
- EComplete Part 4: Execute the Phase 2 Capstone Coding Challenge, training a production-grade Random Forest Regressor and validating stability via 5-Fold Cross-Validation

This assessment is designed to verify your mastery of applied machine learning engineering principles from first principles.

### Phase 2 Examination Structure

### The examination is divided into four sections:

**Part 1:** First-Principles Architecture & Theory (Conceptual Mastery)

**Part 2:** Model Evaluation & Metrics Audit (Analytical Rigor)

**Part 3:** End-to-End ML System Design (Pipeline & Production Best Practices)

**Part 4:** The Phase 2 Capstone Coding Challenge (Practical Implementation)

## Part 1: First-Principles Architecture & Theory

### 1. The Bias-Variance Tradeoff: 

Explain mathematically and intuitively why a model with extremely low training error (zero bias) often suffers from catastrophic failure on test data (high variance). How do ensemble methods like Random Forests specifically mitigate this?

### 2. Linear vs. Logistic Regression: 

Even though logistic regression contains the word "regression," it is a classification algorithm. Explain the role of the Sigmoid function in bridging linear equations to probabilistic classification, and state why standard linear regression fails when applied to binary target labels ($0$ and $1$).

### 3. Tree Splitting Criteria: 

When a Decision Tree selects a feature split at a node, what does Gini Impurity 

## Part 1 Answer:


### 1. The Bias-Variance Tradeoff:

**1. Mathematical Decomposition of Generalization Error**

For a regression task where we attempt to predict a true target $y = f(x) + \epsilon$ (with irreducible noise $\epsilon$ having mean zero and variance $\sigma^2$), the expected mean squared error (MSE) of a trained model $\hat{f}(x)$ at a specific input point $x$ can be rigorously decomposed into three distinct components:$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \text{Bias}(\hat{f}(x))^2 + \text{Var}(\hat{f}(x)) + \sigma^2$$
Where:
- Irreducible Error ($\sigma^2$): The inherent noise in the data generating process that no model can eliminate.
- Bias Squared ($\text{Bias}(\hat{f}(x))^2$): The error introduced by using a model that is too simple to approximate the true underlying function $f(x)$ (underfitting).
- Variance ($\text{Var}(\hat{f}(x))$): How much the model's predictions fluctuate across different training realizations (datasets drawn from the same population).

**2. Why Zero Training Error Leads to High Variance (Intuition)**

When you train an unconstrained model—such as a deep decision tree with no depth limit—it possesses enough capacity to memorize the training data entirely.
- The Path to Zero Bias: By carving up the feature space into microscopic hyperrectangles, the tree can isolate individual data points, driving training error (and therefore squared bias) to $0$. The model assumes the shape of every idiosyncrasy, outlier, and sampling artifact in your training set.
- The Catastrophic Cost (High Variance): Because the model has learned the noise rather than the underlying signal, it becomes hyper-sensitive to the specific sample it was trained on. If you draw a slightly different training sample from the same retail distribution, the decision boundaries will shift violently.
- Test Performance: When exposed to unseen test data, these erratic boundaries cut through unobserved regions incorrectly, causing prediction error to skyrocket. The model has high predictive power over the past, but zero generalization power over the future.

**3. How Random Forests Mitigate High Variance**

Ensemble methods—specifically Random Forests—conquer this trade-off by combining two powerful statistical mechanics: **Bagging (Bootstrap Aggregating)** and **Feature Randomness (Random Subspace Method)**.

**A. The Variance Reduction Equation**

If you average $B$ identically distributed, fully grown decision trees, each having variance $\sigma^2$ and a pairwise correlation $\rho$ between any two trees, the variance of the combined ensemble is given by:$$\text{Var}\left(\frac{1}{B}\sum_{b=1}^B T_b(x)\right) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$
- As the number of trees $B$ approaches infinity, the second term ($\frac{1 - \rho}{B} \sigma^2$) vanishes.
- The remaining variance of the ensemble is strictly bounded by $\rho \sigma^2$.

**B. Decorrelating Trees via Feature Subsampling**

The magic of a Random Forest lies in minimizing the correlation coefficient $\rho$.
- If every tree in the forest considers every feature at every split point, all trees will latch onto the strongest dominant feature (e.g., your primary sales lag), making the trees highly correlated ($\rho \approx 1$), which offers minimal variance reduction over a single tree.
- By enforcing random feature sampling at every node split (e.g., max_features = 'sqrt' or a fraction of total features), the forest forces individual trees to build diverse, orthogonal pathways. Tree #1 might split on price and seasonality first, while Tree #2 splits on transport costs and secondary lags.
- Because the trees become uncorrelated ($\rho \to 0$), averaging their predictions violently cancels out individual tree errors. The ensemble retains the low bias of deep trees while systematically collapsing their high variance into a stable, robust predictor.


### 2. Linear vs. Logistic Regression:

The core distinction lies in how the linear combination of features is transformed to constrain outputs into valid probabilistic boundaries.

**1. Why Standard Linear Regression Fails on Binary Targets ($0$ and $1$)**

When attempting to predict a binary outcome $y \in \{0, 1\}$ using Ordinary Least Squares (OLS) linear regression, the model estimates a continuous line or hyperplane:$$\hat{y} = \mathbf{w}^T \mathbf{x} + b$$

This approach breaks down for three fundamental statistical reasons:

- Unbounded Output Space: Linear regression outputs span the entire real number line $(-\infty, +\infty)$. However, probabilities must be strictly bounded between $0$ and $1$. A linear model frequently predicts nonsensical probabilities like $\hat{y} = 1.4$ (140% probability) or $\hat{y} = -0.3$ (-30% probability).Violation of Error
- Assumptions (Heteroscedasticity): OLS assumes that error terms (residuals) are normally distributed with constant variance (homoscedasticity). For a binary target, the error follows a Bernoulli distribution where the variance is given by $\text{Var}(Y\vert{}X) = p(x)(1 - p(x))$. Because variance depends directly on the predicted probability $p(x)$, the errors are inherently heteroscedastic, rendering standard hypothesis testing and confidence intervals invalid.
- Sensitivity to Outliers and Distorted Decision Boundaries: In linear regression, minimizing squared error heavily penalizes correct predictions that happen to be far past the threshold (e.g., predicting $2.0$ for a true label of $1.0$). Consequently, an outlier in the feature space can dramatically tilt the regression line, shifting the $0.5$ decision boundary and causing systemic misclassifications.

**2. The Role of the Sigmoid Function in Logistic Regression**

To bridge the gap between a continuous linear equation and a bounded probability, logistic regression passes the linear output through a non-linear Sigmoid (Logistic) function.Mathematical FormulationThe sigmoid function is defined as:$$\sigma(z) = \frac{1}{1 + e^{-z}}$$Where $z$ is the standard linear combination (often called the logit or log-odds):$$z = \mathbf{w}^T \mathbf{x} + b$$

**How It Bridges Linear Equations to Probabilities**

Squashing Range to $(0, 1)$:
    - As the linear score $z$ approaches positive infinity ($z \to +\infty$), the term $e^{-z} \to 0$, driving $\sigma(z) \to 1$.
    - As $z$ approaches negative infinity ($z \to -\infty$), the term $e^{-z} \to \infty$, driving $\sigma(z) \to 0$.
    - At exactly $z = 0$, $e^0 = 1$, yielding $\sigma(0) = 0.5$ (the exact decision boundary for binary classification).
- Modeling Log-Odds:
Rather than predicting the probability $p$ directly with a straight line, logistic regression models the log-odds (the logarithm of the odds ratio) linearly:$$\ln\left(\frac{p}{1 - p}\right) = \mathbf{w}^T \mathbf{x} + b$$Applying the sigmoid function is mathematically equivalent to taking the inverse of this logit transformation, converting the unbounded log-odds back into a proper probability $p = P(y=1 \vert{} \mathbf{x})$.


### 3. Tree Splitting Criteria:

**1. What Gini Impurity Measures**

Mathematically, Gini impurity measures the probability that a randomly chosen element from a node would be incorrectly labeled if it were randomly labeled according to the actual class distribution of that node.The Gini impurity $I_G(t)$ at a given node $t$ is calculated as:$$I_G(t) = 1 - \sum_{i=1}^{C} p_i^2$$

Where:

- $C$ is the total number of classes.
- $p_i$ is the probability (or relative frequency) of a sample belonging to class $i$ in node $t$.

**Intuition Behind the Score:**

- Maximum Impurity (Maximum Mixedness): If a binary classification node has a 50/50 split ($p_1 = 0.5, p_2 = 0.5$), the Gini impurity is:$$1 - (0.5^2 + 0.5^2) = 1 - (0.25 + 0.25) = 0.5$$
- This represents the highest possible uncertainty and impurity for a binary split.
- Minimum Impurity: As a node becomes dominated by a single class, the Gini score drops toward zero.

When evaluating a potential feature split, the tree calculates the information gain (or Gini gain)—the weighted reduction in Gini impurity from the parent node to the resulting child nodes—and chooses the split that maximizes this reduction.

**2. What a Gini Score of $0.0$ Signifies**

A Gini score of $0.0$ in a resulting child node signifies absolute purity (homogeneity).
- What it means for the data: Every single data point that landed in that specific child node belongs to the exact same class. There is zero class overlap.
- Algorithmic behavior: When a child node reaches a Gini score of $0.0$, it becomes a pure leaf node (unless stopped by hyperparameters like max_depth or min_samples_split). The tree no longer needs to split this node any further because it can make a prediction with 100% certainty for any sample reaching it.

## Part 2: Model Evaluation & Metrics Audit

### Analyze the following scenario and provide your engineering recommendation:

### Scenario: 

You are evaluating a classification model built to detect fraudulent bank transactions in Lagos. Your dataset is heavily imbalanced: out of 50,000 transactions, only 250 are fraudulent ($0.5\%$). Your model achieves an overall Accuracy of 99.5%.

### The Question: 

Why is accuracy a dangerously misleading metric in this scenario? Which two specific metrics from your evaluation toolkit (from Day 61) should you prioritize, and how do trade-offs between them impact your business stakeholders?

## Part 2 Answer:


In an enterprise fraud detection environment—such as a banking system operating in Lagos—relying on Accuracy is a classic trap that can mask complete operational failure.

**1. Why Accuracy is Dangerously Misleading**

With an extreme class imbalance where only 0.5% of transactions are fraudulent (250 out of 50,000), a naive model that completely fails to detect fraud can still achieve 99.5% Accuracy.
- The Null Model Fallacy: If a model simply predicts 0 (Legitimate) for every single transaction without running any machine learning, it will be correct 49,750 times out of 50,000.
- Asymmetric Cost Blindness: Accuracy treats all errors equally. In fraud detection, however, a False Negative (missing a fraudulent transaction) costs real money, chargebacks, and compliance penalties, while a False Positives (blocking a legitimate customer's transaction) damages customer trust. Accuracy completely obscures this operational reality.

**2. The Two Priority Metrics: Precision and Recall**

A. Recall (Sensitivity / True Positive Rate)$$\text{Recall} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Negatives (FN)}}$$
- What it measures: Out of all actual fraudulent transactions that occurred in the system, what percentage did the model successfully catch?
- Why it matters: In fraud detection, missing a fraud case directly impacts the bank's bottom line and exposes the institution to regulatory sanctions. High recall ensures that fraudulent actors are intercepted.

B. Precision (Positive Predictive Value)$$\text{Precision} = \frac{\text{True Positives (TP)}}{\text{True Positives (TP)} + \text{False Positives (FP)}}$$
- What it measures: Out of all the transactions the model flagged as fraudulent, what percentage were actually fraud?
- Why it matters: Precision measures operational efficiency. If precision is low, the model generates a high volume of false alarms, flooding the fraud operations team with noise.

**3. Business Stakeholder Impact & Trade-Offs**

Because Precision and Recall exist in an inverse relationship (as you tune the decision threshold), balancing them involves a direct trade-off between financial risk and customer experience:
- Optimizing for Recall (The Aggressive Approach):The Tuning: Lowering the decision threshold (e.g., flagging any transaction with $>20\%$ fraud probability).
    - Stakeholder Impact: The fraud team catches almost all fraudulent rings, minimizing financial loss. However, False Positives skyrocket. Legitimate customers in Lagos attempting to buy groceries or pay utility bills get their cards abruptly declined. 
    - This causes immense customer friction, spikes user churn, and overwhelms customer support lines with unblocking requests.
- Optimizing for Precision (The Conservative Approach):
    - The Tuning: Raising the decision threshold (e.g., flagging only transactions with $>80\%$ fraud probability).
    - Stakeholder Impact: Customers experience frictionless, uninterrupted transactions, and the manual review queue stays clean. However, False Negatives spike. Sophisticated fraudsters slip through undetected, leading to severe financial leakage, direct fraud losses, and potential compliance audits from regulatory bodies.

**Engineering Recommendation**

Rather than picking a single fixed threshold, data science and risk teams must plot a Precision-Recall Curve (PR-Curve) and set the operational threshold dynamically based on the transaction type, amount, and customer risk tier. For instance, high-value transfers or international card usage can tolerate a lower threshold (higher recall sensitivity), while routine micro-transactions require higher precision to protect user experience.

## Part 3: End-to-End ML System Design

### Reflecting on your work across Days 64 through 69 on the Nigeria Retail Demand Predictor, address these system architecture points:

### 1. Data Leakage Prevention: 

In your preprocessing pipeline (Day 65), why is it mandatory to split your data into training and test sets before fitting your StandardScaler and OneHotEncoder? What would happen to your test set evaluation if scaling was applied to the entire dataset prior to splitting?

### 2. Uncertainty Quantification: 

On Day 68, we calculated prediction confidence intervals using the standard deviation of test residuals ($\sigma_{\text{residuals}}$). In a supply chain environment (such as managing retail inventory across Kano and Port Harcourt), why is a confidence range superior to a single point prediction?

### 3. Model Explainability: 

Why is impurity-based feature importance insufficient on its own for high-stakes business decisions, and how do feature importance scores help bridge the gap between black-box models and regional inventory managers?

## Part 3 Answer:

### 1. Data Leakage Prevention: 

Splitting your dataset into training and test sets before instantiating and fitting preprocessing transformers is a foundational engineering practice designed to prevent data leakage.

**1. Why Splitting Before Fitting is Mandatory**

In scikit-learn, preprocessing transformers operate in two distinct phases:
- `.fit()` (Parameter Learning): The transformer scans the data to compute and memorize statistical parameters (e.g., StandardScaler calculates the mean $\mu$ and standard deviation $\sigma$; OneHotEncoder builds a vocabulary of valid categories).
`.transform()` (Application): The transformer applies those memorized parameters to scale or encode the dataset.

If you fit your preprocessor on the entire dataset before splitting, you violate the core rule of machine learning validation: the test set must remain completely unobserved and pristine, acting as a strict proxy for future, unseen production data.

**2. What Happens to Your Test Set If You Scale Globally?**

If scaling is applied to the entire dataset prior to splitting, two critical failures occur:

**A. Information Leakage (Target & Feature Contamination)**

When `StandardScaler().fit(X)` runs on the full dataset, the computed mean ($\mu$) and standard deviation ($\sigma$) incorporate the data distribution of the test split (the 20% held-out data).
    - Consequently, your training data features are mathematically adjusted using values derived from your test data.
    - The test set is no longer out-of-sample; it has indirectly "leaked" its statistical properties into the training environment.

**B. Artificially Inflated Evaluation Metrics (Illusory Generalization)**

Because the test set's distribution was baked into the training scaler, your model's evaluation metrics (such as RMSE or $R^2$) on the test set will look deceptively clean and optimistic.
    - The model performs well on the test set not because it genuinely generalized to unseen patterns, but because the test data was already partially "known" to the preprocessing pipeline during training.

**C. Production Failure and Deployment Drift**

In a real-world enterprise deployment, incoming live retail data arrives sequentially or in batches. You cannot compute a global mean and standard deviation across future, unobserved demand data.
    - If your model expects inputs scaled using parameters derived from a historical dataset combined with future test points, live inference requests will experience a distribution mismatch.
    - The production pipeline will be forced to use immutable parameters locked down from the training set alone (via `.transform()`), causing severe prediction errors if the training scaler was originally contaminated by global fitting.


### 2. Uncertainty Quantification: 

In enterprise retail supply chain management—especially when coordinating multi-node fulfillment across distinct commercial hubs like Kano (a major northern trading and agricultural aggregation hub) and Port Harcourt (a southern coastal and industrial port gateway)—relying on a deterministic point prediction is an operational risk.

While a model might output a single point estimate (e.g., predicting exactly 1,250 units of demand for a specific SKU next week), real-world retail demand is stochastic. A confidence range (derived from test residuals via $\sigma_{\text{residuals}}$) is superior for several critical operational reasons:

**1. Accounting for Asymmetric Business Costs (Stockouts vs. Holding Costs)**
In retail, the financial penalty of a prediction error is rarely symmetric:
- Under-forecasting (False Negative): Results in stockouts, lost revenue, damaged customer loyalty, and empty shelves.
- Over-forecasting (False Positive): Results in tied-up working capital, excess warehouse holding costs, and potential dead stock (especially hazardous for perishables or fast-moving consumer goods).

A point prediction gives inventory planners a single target, forcing them to guess the risk margin. A 95% confidence interval provides the upper and lower bounds, allowing logistics managers to adjust inventory policies dynamically based on item margin. For high-margin items, planners can bias stocking toward the upper bound to prevent stockouts; for low-margin or bulky goods, they can anchor closer to the lower bound.

**2. Direct Integration into Safety Stock and Reorder Point Formulas**

Supply chain optimization cannot function on averages alone; it requires variance modeling to survive disruptions. The standard deviation of your test residuals ($\sigma_{\text{residuals}}$) is the exact mathematical bridge needed to calculate Safety Stock ($SS$):$$\text{Safety Stock} = Z \cdot \sigma_{\text{residuals}} \cdot \sqrt{\text{Lead Time}}$$

Without calculating residual standard deviation, automated procurement systems cannot determine how many buffer units to hold to maintain a 95% or 99% service level.The confidence margin calculated on Day 68 transforms a machine learning metric into an actionable inventory control parameter.

**3. Hedging Against Regional and Infrastructural Volatility**

Kano and Port Harcourt present vastly different supply chain profiles—ranging from northern seasonal harvest cycles and cross-border trade flows to southern maritime logistics, port clearance times, and regional purchasing power fluctuations.
    - A single point prediction assumes uniform certainty across both regions.
    - In reality, transport bottlenecks, fuel price spikes, or regional demand shocks mean prediction error ($\sigma$) will fluctuate.

Evaluating the residual distribution allows enterprise stakeholders to see that a forecast for Kano might carry a wider confidence band (higher variance due to localized supply chain variables) than a stable urban center in Lagos, enabling region-specific risk mitigation.

**4. Executive Decision-Making and Risk Mitigation**

Enterprise stakeholders (such as supply chain directors, CFOs, and procurement leads) are accountable for capital allocation. When presented with a capital-intensive procurement plan, executives cannot make sound financial commitments based on an unverified single number.

Providing a 95% confidence interval equips leadership with a worst-case and best-case financial scenario. Knowing that demand will fall between an empirically bounded range allows the business to secure flexible supplier contracts, optimize warehouse leasing, and maintain adequate cash flow without flying blind.

### 3. Model Explainability:

While impurity-based feature importance scores (like Mean Decrease in Impurity / MDI extracted from Random Forests) provide a useful high-level diagnostic during model development, relying on them as the sole bridge for high-stakes enterprise decisions introduces significant operational risk.

**1. Why Impurity-Based Feature Importance is Insufficient on Its Own**

Impurity-based metrics measure how effectively a feature splits nodes across training data trees, but they suffer from structural biases that make them unreliable for executive and operational decision-making:
- Cardinality and Scale Bias: MDI inherently favors continuous numerical features (such as price_ngn or historical sales lag values) over sparse binary indicators (such as holiday flags, regional tags, or payday markers). Because continuous features offer many more potential split thresholds, they accumulate higher impurity reductions artificially, often masking the true importance of low-cardinality business drivers.
- In-Sample vs. Generalization Mismatch: MDI is calculated entirely on the training dataset. It reflects what the trees learned to memorize or split on during training, rather than how reliably a feature generalizes to unseen test data or shifting production distributions.
- Collinearity Distortions: In retail supply chains, features are frequently correlated (e.g., fuel transport costs moving in tandem with wholesale prices). When collinear features compete for splits, MDI splits the importance credit arbitrarily between them, making feature rankings unstable and misleading for analysts.
- The Global vs. Local Blindspot: MDI provides a global ranking across the entire forest. However, a supply chain director or regional planner does not make decisions on a global average; they need to know what drove a specific prediction for a specific SKU in a specific warehouse at a specific time. Global impurity metrics provide zero visibility into local predictions.

**2. Bridging the Gap Between Black-Box Models and Regional Inventory Managers**

To successfully deploy machine learning models in enterprise environments—spanning diverse economic hubs like the agricultural and trading markets of Kano and the industrial port logistics of Port Harcourt—feature importances must be translated into actionable operational insights. Here is how structured feature importance and explainability frameworks bridge that gap:

**A. Validating Domain Intuition (Establishing Baseline Trust)**

Before regional planners can trust a model, they need to know it isn't learning spurious noise. When global feature importance scores confirm that historical sales lags and price elasticity rank highest, it aligns with the planner's decades of domain expertise. This initial validation breaks down skepticism: planners see that the model respects core retail realities rather than relying on random statistical artifacts.

**B. Transitioning from Global Ranking to Local Explainability (SHAP & Permutation)**

To empower regional managers, engineering teams must move beyond MDI and integrate local instance explanations (such as SHAP - Shapley Additive exPlanations).
- Instead of telling a manager in Kano that "price is important globally," a local explanation reveals: "Demand for this SKU is projected to drop 20% this week specifically because the retail price increased by ₦150 relative to historical baselines."
- This translates abstract machine learning metrics into concrete, human-readable causal narratives.

**C. Reconciling Model Predictions with Unquantified Local Anomalies**

Regional managers operate in environments subject to unquantified shocks—such as sudden border policy shifts in the north, regional fuel price hikes, or localized festive gatherings.
- When a model flags an unexpected surge in inventory requirements for Port Harcourt, feature attribution scores allow the manager to inspect which features pushed the prediction high.
- If the manager sees that the model is reacting appropriately to a verified transport cost increase or seasonal flag, they approve the procurement order.
- Conversely, if the attribution reveals the model is latching onto a transient data glitch, the manager has the explainable justification needed to safely override the algorithm.

**C. Fostering Cross-Functional Accountability**

Enterprise governance requires that automated supply chain decisions remain auditable. Feature importance scores and local explanations serve as a shared language between data science teams, financial auditors, and logistics operators. They transform a black-box software recommendation into an transparent, defensible business proposal.

## Part 4: The Phase 2 Capstone Coding Challenge

### To certify your practical readiness for Phase 3, write a clean, production-grade Python script that ties together your learning.

### Challenge Specifications

**Write a script that accomplishes the following:**

- Loads your preprocessed training data (retail_demand_train_processed.csv) and test data (retail_demand_test_processed.csv).
- Trains a Random Forest Regressor with n_estimators=200, max_depth=10, and random_state=42.
- Evaluates the model on the test set, computing MAE, RMSE, and $R^2$ score.
- Performs a 5-Fold Cross-Validation on the training dataset using scoring='r2' to verify that your model's performance is stable and free from sampling bias.Prints a clean, formatted audit summary to the terminal.

In [1]:
# Capstone Template Skeleton
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load data
train_df = pd.read_csv("retail_demand_train_processed.csv")
test_df = pd.read_csv("retail_demand_test_processed.csv")

X_train = train_df.drop(columns=['demand_units'])
y_train = train_df['demand_units']
X_test = test_df.drop(columns=['demand_units'])
y_test = test_df['demand_units']

# 2. Instantiate and train model
rf_model = RandomForestRegressor(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# 3. Test set evaluation
y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# 4. 5-Fold Cross-Validation across training set
# Combine X_train and y_train temporarily if needed, or pass X_train directly to cross_val_score
# Note: For full CV across the pipeline, cross_val_score accepts the model, features, and target.
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2')

# 5. Print Audit Report
print("==================================================")
print("     PHASE 2 CAPSTONE: MODEL AUDIT REPORT         ")
print("==================================================\n")
print(f"Test Set MAE:  {mae:.2f} units")
print(f"Test Set RMSE: {rmse:.2f} units")
print(f"Test Set R²:   {r2:.4f}\n")
print(f"5-Fold CV R² Scores: {np.round(cv_scores, 4)}")
print(f"Mean CV R² Score:    {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

     PHASE 2 CAPSTONE: MODEL AUDIT REPORT         

Test Set MAE:  9.74 units
Test Set RMSE: 14.48 units
Test Set R²:   0.9627

5-Fold CV R² Scores: [0.9498 0.9511 0.9562 0.9555 0.9598]
Mean CV R² Score:    0.9545 (±0.0036)
